# Sham — CPU Training Track

One-time setup:
1. Accelerator: None. Internet: On.
2. Secrets: GITHUB_TOKEN, KAGGLE_USERNAME, KAGGLE_KEY.
3. Create an empty Kaggle Dataset named sham-cpu-track-checkpoint once.
4. First run only: Add Input -> the GPU track's nova-small-checkpoint dataset.
5. Save, then Schedule this notebook (as often as you like, no GPU quota here).

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py"))
sys.path.insert(0, CODE_DIR)
print("Sham code ready at:", CODE_DIR)

In [ ]:
import os
import sys

print("جاري تفعيل الوضع غير المقيد الشامل...")

# 1. ضبط المتغيرات البيئية
os.environ["DISABLE_SAFETY_FILTER"] = "1"
os.environ["DISABLE_CONTENT_FILTER"] = "1"
os.environ["UNRESTRICTED_MODE"] = "1"
os.environ["NO_MODERATION"] = "1"

def force_uncensored(x, *args, **kwargs):
    return x if x is not None else True

# 2. تعيين التوابع العامة
globals()["force_uncensored"] = force_uncensored
globals()["no_filter"] = force_uncensored
globals()["uncensored"] = force_uncensored

# 3. معالجة الموديول والـ Class ديناميكياً
try:
    import dataset
    from dataset import TextSequenceDataset

    methods = ['filter_text', 'is_banned', 'clean_text', 'safety_filter', 'moderate', 'check_content', 'validate_text', 'ban_words']
    
    # تعطيل الدوال داخل الكلاس
    for method_name in methods:
        if hasattr(TextSequenceDataset, method_name):
            setattr(TextSequenceDataset, method_name, lambda self, *args, **kwargs: args[0] if args else True)
            print("تم تعطيل في Class:", method_name)

    # تعطيل الدوال إذا كانت موجودة على مستوى الموديول (dataset.py) مباشرة
    for method_name in methods:
        if hasattr(dataset, method_name):
            setattr(dataset, method_name, force_uncensored)
            print("تم تعطيل في Module:", method_name)

    print("تم تطبيق التعطيل الشامل على TextSequenceDataset و dataset module")
except Exception as e:
    print("لم يتم العثور على TextSequenceDataset:", e)

print("الوضع غير المقيد مفعّل بنجاح والقواعد حُيّدت بالكامل.")


In [ ]:
try:
    import tokenizers
except ImportError:
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
print("tokenizers ready.")

In [ ]:
from data_acquisition import stream_hf_text_corpus

MAX_DOCUMENTS = 5_000

corpus_dir = "/kaggle/working/corpus/wikipedia_ar"
corpus_files = stream_hf_text_corpus(
    dataset_name="wikimedia/wikipedia",
    config_name="20231101.ar",
    text_field="text",
    output_dir=corpus_dir,
    max_documents=MAX_DOCUMENTS,
)
print(f"shard files: {len(corpus_files)}")

In [ ]:
from pathlib import Path
from text_tokenizer import train_text_tokenizer, ShamTextTokenizer
from model import TEXT_VOCAB_SIZE

previous_tokenizer_files = list(Path("/kaggle/input").rglob("*tokenizer*.json"))
if previous_tokenizer_files:
    tokenizer_path = previous_tokenizer_files[0]
    tokenizer = ShamTextTokenizer.load(str(tokenizer_path))
    print(f"reused an existing tokenizer from a previous session/fork: {tokenizer_path}")
else:
    tokenizer = train_text_tokenizer(corpus_files, vocab_size=TEXT_VOCAB_SIZE)
    print("trained a fresh tokenizer (no previous one found -- this should only happen on a true first run).")
tokenizer.save("/kaggle/working/sham_cpu_tokenizer.json")

In [ ]:
import torch
from dataset import TextSequenceDataset

SEQ_LEN = 512
text_dataset = TextSequenceDataset(corpus_files, tokenizer, seq_len=SEQ_LEN)
print(f"real training windows: {len(text_dataset):,}")

In [ ]:
from model import ShamSmallConfig, ShamSmall, TOTAL_VOCAB_SIZE

model_cfg = ShamSmallConfig(
    vocab_size=TOTAL_VOCAB_SIZE, d_model=768, n_layers=12, n_heads=12, n_kv_heads=4,
    mlp_hidden=2048, max_seq_len=SEQ_LEN, use_gradient_checkpointing=True,
)
model = ShamSmall(model_cfg)
print(f"model: {model.count_parameters():,} real parameters.")

device = "cpu"
print(f"device: {device}")

In [ ]:
from checkpoint import load_checkpoint

start_step = 0
resume_optimizer = None

previous_checkpoints = sorted(
    Path("/kaggle/input").rglob("step_*.pt"),
    key=lambda p: int(p.stem.split("_")[1]),
)
if previous_checkpoints:
    last_ckpt = previous_checkpoints[-1]
    model, start_step, _ = load_checkpoint(last_ckpt, map_location=device)
    from train import build_optimizer
    resume_optimizer = build_optimizer(model, lr=3e-4, weight_decay=0.1)
    load_checkpoint(last_ckpt, map_location=device, load_optimizer_into=resume_optimizer)
    print(f"resumed from: {last_ckpt} (step {start_step:,})")
else:
    print("no previous checkpoint found -- starting from scratch (expected only on a true first run).")

In [ ]:
import time
from train import TrainConfig, build_optimizer, build_lr_scheduler

CALIBRATION_STEPS = 10
calib_batches = [
    torch.stack([text_dataset[i] for i in range(b, b + 2)])
    for b in range(0, min(len(text_dataset) - 2, CALIBRATION_STEPS * 2 * 4), 2)
][: CALIBRATION_STEPS * 4]
assert calib_batches, "not enough data to calibrate -- increase MAX_DOCUMENTS above."

model.to(device)
model.train()
_calib_optimizer = resume_optimizer or build_optimizer(model, lr=3e-4, weight_decay=0.1)

t0 = time.time()
steps_done = 0
for batch in calib_batches:
    batch = batch.to(device)
    _, loss = model(batch, labels=batch)
    loss.backward()
    _calib_optimizer.step()
    _calib_optimizer.zero_grad()
    steps_done += 1
    if steps_done >= CALIBRATION_STEPS:
        break
elapsed = time.time() - t0
steps_per_second = steps_done / elapsed

MAX_TRAINING_HOURS = 8.5
realistic_steps_for_session = max(int(steps_per_second * MAX_TRAINING_HOURS * 3600 * 0.85), 20)

print(f"real measured speed: {steps_per_second:.4f} steps/sec on {device}")
print(f"realistic steps for this session (with safety margin): {realistic_steps_for_session:,}")

In [ ]:
TOTAL_STEPS = realistic_steps_for_session
num_windows = len(text_dataset) - (len(text_dataset) % 4)

def _batch_iterator():
    while True:
        for b in range(0, num_windows, 4):
            yield torch.stack([text_dataset[i] for i in range(b, b + 4)])

import itertools
batches = itertools.islice(_batch_iterator(), TOTAL_STEPS)

train_cfg = TrainConfig(
    seq_len=SEQ_LEN,
    batch_size=4,
    grad_accum_steps=4,
    lr=3e-4,
    warmup_steps=max(20, TOTAL_STEPS // 100),
    total_steps=start_step + TOTAL_STEPS,
    checkpoint_dir="/kaggle/working/checkpoints",
    checkpoint_every=100,
    log_every=10,
    max_wall_clock_seconds=MAX_TRAINING_HOURS * 3600,
)

from train import train
loss_history = train(
    model, batches, train_cfg, device=device,
    start_step=start_step, resume_optimizer=resume_optimizer or _calib_optimizer,
)
print(f"\nreal steps this session: {len(loss_history):,}")
if loss_history:
    print(f"first 10 avg loss: {sum(loss_history[:10]) / min(10, len(loss_history)):.4f}")
    print(f"last 10 avg loss: {sum(loss_history[-10:]) / min(10, len(loss_history)):.4f}")

In [ ]:
from checkpoint import save_checkpoint

final_step = start_step + len(loss_history)
save_checkpoint("/kaggle/working/checkpoints/final.pt", model, final_step)
print(f"final checkpoint saved locally at step {final_step:,}.")

In [ ]:
import json as _json
import shutil as _shutil

subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=False)

KAGGLE_USERNAME = UserSecretsClient().get_secret("KAGGLE_USERNAME")
KAGGLE_KEY = UserSecretsClient().get_secret("KAGGLE_KEY")
DATASET_SLUG = f"{KAGGLE_USERNAME}/sham-cpu-track-checkpoint-v2"

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

upload_dir = Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    _shutil.rmtree(upload_dir)
(upload_dir / "checkpoints").mkdir(parents=True)
for ckpt in Path("/kaggle/working/checkpoints").glob("*.pt"):
    _shutil.copy2(ckpt, upload_dir / "checkpoints" / ckpt.name)
_shutil.copy2("/kaggle/working/sham_cpu_tokenizer.json", upload_dir / "sham_cpu_tokenizer.json")

metadata = {"title": "sham-cpu-track-checkpoint-v2", "id": DATASET_SLUG, "licenses": [{"name": "unknown"}]}
(upload_dir / "dataset-metadata.json").write_text(_json.dumps(metadata))

_list_result = subprocess.run(["kaggle", "datasets", "list", "-m", "--csv"], capture_output=True, text=True)
_dataset_exists = DATASET_SLUG in (_list_result.stdout or "")

if _dataset_exists:
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(upload_dir), "-m", f"cpu-track auto-update at step {final_step:,}", "-r", "zip"],
        capture_output=True, text=True,
    )
else:
    result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(upload_dir), "-r", "zip"],
        capture_output=True, text=True,
    )
_combined = (result.stdout or "") + (result.stderr or "")

if result.returncode == 0 and "error" not in _combined.lower():
    verb = "published to" if _dataset_exists else "created"
    print(f"{verb} {DATASET_SLUG} at step {final_step:,} -- the next scheduled run will pick it up automatically.")
elif "incompatible" in _combined.lower():
    print("WARNING: incompatible dataset. Bump -v2 to -v3 and re-run.")
    print(_combined)
else:
    print("WARNING: failed to publish -- checkpoint safe in Output.", _combined)